In [8]:
import numpy as np
import matplotlib.pyplot as plt
import allantools 
import pandas as pd 

# Define functions

In [9]:
def absln(x):
    if x == 0:
        return 0
    else:
        return np.log(np.abs(x))

In [10]:
def s_w(t,alpha):
    
    values = {2:-np.abs(t), \
              1:t**2*absln(t), \
              0:np.abs(t)**3, \
              -1:-t**4*absln(t), \
              -2:-np.abs(t)**5, \
              -3:t**6*absln(t), \
              4: np.abs(t)**7}
    
    if alpha in values.keys():
        return values[alpha]
    else:
        print('Choose a value for alpha depending on the noise type')

In [11]:
def s_x(t,F,alpha):
    if F == np.inf and alpha in range(-4,1):
        return s_w(t,alpha+2)
    else:
        return F**2*(2*s_w(t,alpha)-s_w(t-1/F,alpha)-s_w(t+1/F,alpha))

In [12]:
def s_z(t,F,alpha,d):
    if d == 1:
        return 2*s_x(t,F,alpha)-s_x(t-1,F,alpha)-s_x(t+1,F,alpha)
    elif d == 2: 
        return 6*s_x(t,F,alpha)-4*s_x(t-1,F,alpha)-4*s_x(t+1,F,alpha)+s_x(t-2,F,alpha)+s_x(t+2,F,alpha)
    elif d == 3:
        return 20*s_x(t,F,alpha)-15*s_x(t-1,F,alpha)-15*s_x(t+1,F,alpha)+6*s_x(t-2,F,alpha)+6*s_x(t+2,F,alpha)-s_x(t-3,F,alpha)-s_x(t+3,F,alpha)
    else:
        print('Choose a value for d depending on the type of variance')

In [13]:
def BasicSum(J,M,S,F,alpha,d):
    return s_z(0,F,alpha,d)**2 +(1-J/M)*(s_z(J/S,F,alpha,d))**2+2*sum((1-j/M)*(s_z(j/S,F,alpha,d))**2 for j in range(1,J))

# Initial parameters

In [14]:
Jmax = 100

In [15]:
L = lambda m,F,d: m/F+m*d

In [16]:
M = lambda m,d,N,S,F: 1+np.floor(S*(N-L(m,F,d))/m)

In [17]:
J = lambda m,d,N,S,F: min(M(m,d,N,S,F),(d+1)*m)

In [18]:
r = lambda m,d,N,S,F: M(m,d,N,S,F)/S

In [19]:
coef_unmod_allan = {2:(35/18,1), \
                  1:(790,410), \
                  0:(2/3,1/3), \
                  -1:(0.852,0.375), \
                  -2:(1.079,0.368), \
                  -3:(0,0), \
                  4: (0,0)}

In [20]:
coef_unmod_allan[2][0]

1.9444444444444444

In [73]:
def case_two(J,Jmax,m,alpha,d,r,N,S,F,M):
    if J <= Jmax:
        if m*(d+1)<=Jmax:
            mp = m 
        else:
            mp = np.inf        
        return 'Case of J<= Jmax: 1/edf = ', 1/((s_z(0,mp,alpha,d))**2)*BasicSum(J,M,S,mp,alpha,d)
    elif J > Jmax and r >= d+1:
        a0 = coef_unmod_allan[alpha][0]
        a1 = coef_unmod_allan[alpha][1]
        return 'Case of J > Jmax: 1/edf = ', 1/r*(a0-a1/r)
    else:
        mp = Jmax/r
        return 'Else case: 1/edf = ', 1/((s_z(0,np.inf,alpha,d))**2)*BasicSum(Jmax,Jmax,mp,np.inf,alpha,d)

## Case of J<= Jmax

In [33]:
mc = 1
dc = 2
Nc = 1000

In [34]:
M(mc,dc,Nc,mc,mc)

998.0

In [32]:
J(mc,dc,Nc,mc,mc)

3

In [27]:
r(mc,dc,Nc,mc,mc)

998.0

In [72]:
case_two(J(mc,dc,Nc,mc,mc),Jmax,mc,0,dc,r(mc,dc,Nc,mc,mc),Nc,mc,mc,M(mc,dc,Nc,mc,mc))

('Case of J<= Jmax: 1/edf = ', 1.2774437764417723)

# Case of J > Jmax

In [60]:
md = 100
Nd = 1000

In [61]:
J(md,dd,Nd,md,md)

300

In [64]:
BasicSum(J(md,dc,Nd,md,md),M(md,dc,Nd,md,md),md,md,0,dc)

35995.58620005071

In [71]:
case_two(J(md,dc,Nd,md,md),Jmax,md,0,dc,r(md,dc,Nd,md,md),Nd,md,md,M(md,dd,Nd,md,md))

('Case of J > Jmax: 1/edf = ', 0.078125)

# Else case

In [76]:
s_z(0,np.inf,1,2)

/home/anto/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:5: RuntimeWarning: invalid value encountered in double_scalars
  """


nan

In [82]:
s_x(1,np.inf,1)

/home/anto/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:5: RuntimeWarning: invalid value encountered in double_scalars
  """


nan

# Using Tweezepy

In [86]:
from tweezepy import allanvar

In [87]:
dir(allanvar)

['__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__spec__',
 'avar',
 'calc_avar',
 'calc_totvar',
 'edf_approx',
 'edf_greenhall',
 'edf_simple',
 'edf_totdev',
 'greenhall_BasicSum',
 'greenhall_sw',
 'greenhall_sx',
 'greenhall_sz',
 'greenhall_table1',
 'greenhall_table2',
 'greenhall_table3',
 'm_generator',
 'noise_id',
 'np',
 'scipy',
 'totvar',
 'totvar_bias',
 'warn']